<a href="https://colab.research.google.com/github/svarshney25/Solar-Effects/blob/main/CME_and_proton_temps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
#import matlab and set inline plotting
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
#import os and setup file things
from os import path as path
data_folder = '/content/drive/MyDrive/mySEES/Proton_temp_SEES_2024/'
data_folder2018 = path.join(data_folder, 'CME_2018')




ValueError: mount failed

In [ ]:
#read in mars data
mars_data_path = path.join(data_folder2018, 'MAVEN_SWIA_Mars.csv')
print(mars_data_path)
mars_data = pd.read_csv(mars_data_path)
mars_data["UT"] = pd.to_datetime(mars_data["UT"], format="%Y-%m-%d/%H:%M:%S")
print(mars_data.columns)



In [ ]:

# Load the CSV Data
data = pd.read_csv(mars_data_path, parse_dates=['UT'])

# Preprocess the Data
#fills an missing data points with the points directly before it
data.fillna(method='ffill', inplace=True)

# Step 3: Identify CME Signatures
proton_number = data.iloc[:,1]
velocity = data.iloc[:,2]
temperature = data.iloc[:,3]

# Define thresholds for sudden changes
#sets CME threshold to be three standard deviations above mean: equivalent to the 99.7% percentile
density_threshold = proton_number.mean() + 2.5 * proton_number.std()
velocity_threshold = velocity.mean() + 2.5 * velocity.std()
temperature_threshold = temperature.mean() + 2.5 * temperature.std()

# Detect CME Events
#creates a new collumn that is a boolean array, where true marks a CME event and false is otherwise
data['CME_event'] = (
    (proton_number > density_threshold).astype(int) +
    (velocity > velocity_threshold).astype(int) +
    (temperature > temperature_threshold).astype(int)
) > 1

# plot the data
plt.figure(figsize=(15, 10))

plt.subplot(3, 1, 1)
plt.plot(data['UT'], proton_number, label='Proton Number')
plt.axhline(density_threshold, color='r', linestyle='--', label='Density Threshold')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(data['UT'], velocity, label='Velocity')
plt.axhline(velocity_threshold, color='r', linestyle='--', label='Velocity Threshold')
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(data['UT'], temperature, label='Temperature')
plt.axhline(temperature_threshold, color='r', linestyle='--', label='Temperature Threshold')
plt.legend()

plt.tight_layout()
plt.show()

# Print CME event times
cme_times = data[data['CME_event']]['UT']
print("CME events detected at the following times:")
print(cme_times)


Plot the data from the earth sattelites:


In [ ]:
Earth_data_DSCOVR_path = path.join(data_folder2018, 'DSCOVR_temperature_Earth.csv')
Earth_data_GOES_path = path.join(data_folder2018, 'GOES_flux_Earth.csv')
Earth_data_ACE_path = path.join(data_folder2018, 'ACE_density_Earth.csv')
Earth_data_KPindex_path = path.join(data_folder2018, 'NOAA_KP_Earth.csv')

# get and preprocess data

earth_DSCOVR_data = pd.read_csv(Earth_data_DSCOVR_path, parse_dates=[0])
earth_GOES_data = pd.read_csv(Earth_data_GOES_path, parse_dates=[0])
earth_ACE_data = pd.read_csv(Earth_data_ACE_path, parse_dates=[0])
earth_KPindex_data = pd.read_csv(Earth_data_KPindex_path, parse_dates=[0])



#remove any negative values, because non of these quantities can actually be negative
earth_DSCOVR_data = earth_DSCOVR_data[earth_DSCOVR_data.iloc[:,1] > 0]
earth_GOES_data = earth_GOES_data[earth_GOES_data.iloc[:,1] > 0]
earth_ACE_data = earth_ACE_data[earth_ACE_data.iloc[:,1] > 0]
earth_KPindex_data = earth_KPindex_data[earth_KPindex_data.iloc[:,1] > 0]

temp_threshold = earth_DSCOVR_data.iloc[:,1].mean() + 3 * earth_DSCOVR_data.iloc[:,1].std()
flux_threshold = earth_GOES_data.iloc[:,1].mean() + 3 * earth_GOES_data.iloc[:,1].std()
density_threshold = earth_ACE_data.iloc[:,1].mean() + 3 * earth_ACE_data.iloc[:,1].std()
KP_threshold = earth_KPindex_data.iloc[:,1].mean() + 3.5 * earth_KPindex_data.iloc[:,1].std()

earth_DSCOVR_data = earth_DSCOVR_data[earth_DSCOVR_data.iloc[:,1] < temp_threshold]
earth_GOES_data = earth_GOES_data[earth_GOES_data.iloc[:,1] < flux_threshold]
earth_ACE_data = earth_ACE_data[earth_ACE_data.iloc[:,1] < density_threshold]
earth_KPindex_data = earth_KPindex_data[earth_KPindex_data.iloc[:,1] < KP_threshold]

#these lines arent strictly necessary, but they should fill out any missing data with the data points before them. tbh there just there bc chatgpt suggested them
earth_DSCOVR_data.fillna(method='ffill', inplace=True)
earth_GOES_data.fillna(method='ffill', inplace=True)
earth_ACE_data.fillna(method='ffill', inplace=True)
earth_KPindex_data.fillna(method='ffill', inplace=True)



earth_proton_temp = earth_DSCOVR_data.iloc[:,1]
earth_proton_flux = earth_GOES_data.iloc[:,1]
earth_proton_density = earth_ACE_data.iloc[:,1]
earth_KP_index = earth_KPindex_data.iloc[:,1]

#define thresholds for CME values
# in retrospect, I dotn think this is particularly helpful, but im keeping it here justin case it illuminates something


#doing some other filtering here, to remove some outliers, maybe itll be helpful


# detect CME events
earth_DSCOVR_data['CME_event'] = (
    (earth_proton_density > density_threshold).astype(int) &
    (earth_proton_temp > temp_threshold).astype(int) &
    (earth_proton_flux > flux_threshold).astype(int) &
    (earth_KP_index > KP_threshold).astype(int)

)

#plot data

plt.figure(figsize=(15, 10))

plt.subplot(4, 1, 1)
plt.plot(earth_DSCOVR_data.iloc[:,0], earth_proton_temp, label='Proton temperature')
#plt.axhline(temp_threshold, color='r', linestyle='--', label='temperature Threshold')
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(earth_GOES_data.iloc[:,0], earth_proton_flux, label='proton flux')
#plt.axhline(flux_threshold, color='r', linestyle='--', label='p_flux Threshold')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(earth_ACE_data.iloc[:,0], earth_proton_density, label='proton density')
#plt.axhline(density_threshold, color='r', linestyle='--', label='p_density Threshold')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(earth_KPindex_data.iloc[:,0], earth_KP_index, label='KP index')
#plt.axhline(KP_threshold, color='r', linestyle='--', label='KP Threshold')
plt.legend()

plt.tight_layout()
plt.show()

# Print CME event times
#cme_times = data[data['CME_event']]['UT']
#print("CME events detected at the following times:")
#print(cme_times)








I'm not getting anything particularly enlightening from the above polotted data. In the next section im going to take a local average of each data point in order to smooth the data. Then plot the times the CME's hit earth based on data from the paper

In [ ]:
import datetime


def local_average(data, rmean):
  new_data = data.copy()
  i = rmean
  while i < (len(data) - rmean):
    new_data.iloc[i] = data.iloc[i-rmean:i+rmean].mean()
    i += 1
  return new_data

avged_earth_proton_temp = local_average(earth_proton_temp, 30)
avged_earth_proton_flux = local_average(earth_proton_flux, 30)
avged_earth_proton_density = local_average(earth_proton_density, 30)
avged_earth_KP_index = local_average(earth_KP_index, 30)


In [ ]:

#recorded CME start times from the paper: (I eyeballed these from the graphs)

CME1_shock1 = datetime.datetime(2018,8,24,6, tzinfo=datetime.timezone.utc)
CME1_start_time = datetime.datetime(2018,8,24,11, tzinfo=datetime.timezone.utc)
CME1_shock2 = datetime.datetime(2018,8,24,1, tzinfo=datetime.timezone.utc)
CME1_end_time = datetime.datetime(2018,8,25,6, tzinfo=datetime.timezone.utc)
CME2_start_time = datetime.datetime(2018,8,25,13, tzinfo=datetime.timezone.utc)
CME2_end_time = datetime.datetime(2018,8,26,12, tzinfo=datetime.timezone.utc)



plt.figure(figsize=(15, 10))

fig, axs = plt.subplots(4, 1, sharex=False, figsize=(15, 10))


axs[0].plot(earth_DSCOVR_data.iloc[:,0], avged_earth_proton_temp, label='Proton temperature')
axs[1].plot(earth_GOES_data.iloc[:,0], avged_earth_proton_flux, label='proton flux')
axs[2].plot(earth_ACE_data.iloc[:,0], avged_earth_proton_density, label='proton density')
axs[3].plot(earth_KPindex_data.iloc[:,0], avged_earth_KP_index, label='KP index')

for ax in axs:
    ax.axvline(CME1_shock1, color='r', linestyle='--', label='CME1 initial shock')
    ax.axvspan(CME1_start_time, CME1_end_time, color = 'purple', alpha = 0.3, label='CME 1')
    ax.axvspan(CME2_start_time, CME2_end_time, color = 'gray', alpha = 0.3, label='CME 2')
    ax.legend()

plt.tight_layout()
plt.show()

make histograms with standard deviations and means for before during and after

In [ ]:
import inspect

def get_variable_name(variable):
    """
    Retrieves the name of the given variable by inspecting the local variables in the caller's scope.
    """
    frame = inspect.currentframe().f_back.f_back
    local_vars = frame.f_locals
    for var_name, var_value in local_vars.items():
        if var_value is variable:
            return var_name
    return None

def create_histograms(datasets, times_array, start_time, end_time, bins='auto'):
    """
    Create histograms of multiple datasets within a specified time range.

    Parameters:
    - datasets: List of data lists or numpy arrays.
    - times_array: List of lists or numpy arrays of corresponding time values. Should be in the same order as datasets.
    - start_time: Start time for filtering data. Should be in a format compatible with pandas.Timestamp.
    - end_time: End time for filtering data. Should be in a format compatible with pandas.Timestamp.
    - bins: Number of bins or binning strategy for the histogram. Default is 'auto'.

    Returns:
    - None. Displays the histogram plots.
    """
    num_datasets = len(datasets)
    if num_datasets != len(times_array):
        raise ValueError("The number of datasets must match the number of time series arrays.")

    plt.figure(figsize=(10, 6 * num_datasets))

    for i, (data, times) in enumerate(zip(datasets, times_array)):
        # Convert times to pandas datetime if not already
        times = pd.to_datetime(times)

        # Create a DataFrame
        df = pd.DataFrame({'data': data, 'times': times})

        # Filter the DataFrame based on the given time range
        mask = (df['times'] >= pd.to_datetime(start_time)) & (df['times'] <= pd.to_datetime(end_time))
        filtered_data = df.loc[mask, 'data']

        # Get the name of the variable passed as data
        data_name = get_variable_name(datasets[i])
        if not data_name:
            data_name = f'dataset_{i+1}'

        # Plot the histogram
        plt.subplot(num_datasets, 1, i + 1)
        plt.hist(filtered_data, bins=bins, edgecolor='black')
        plt.title(f'Histogram of {data_name} from {start_time} to {end_time}')
        plt.xlabel(data_name)
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

In [ ]:
raw_datasets = [earth_proton_temp, earth_proton_flux, earth_proton_density, earth_KP_index]
times_array = [earth_DSCOVR_data.iloc[:,0], earth_GOES_data.iloc[:,0], earth_ACE_data.iloc[:,0], earth_KPindex_data.iloc[:,0]]
min_max = [(0,600000),(0,),(),()]

create_histograms(raw_datasets, times_array, CME1_shock1, CME2_end_time, bins=30)

In [ ]:
create_histograms(raw_datasets, times_array, datetime.datetime(2018,8,16,tzinfo=datetime.timezone.utc), CME1_shock1, bins=30)

In [ ]:
create_histograms(raw_datasets, times_array, CME2_end_time, datetime.datetime(2018,9,10,tzinfo=datetime.timezone.utc), bins=30)

now all of the same graphs but with the unsmoothed data


In [ ]:
create_histogram(earth_proton_temp, earth_DSCOVR_data.iloc[:,0], CME1_shock1, CME2_end_time, x_label = 'Proton Temperature')
create_histogram(earth_proton_flux, earth_GOES_data.iloc[:,0], CME1_shock1, CME2_end_time, x_label = 'proton flux')
create_histogram(earth_proton_density, earth_ACE_data.iloc[:,0], CME1_shock1, CME2_end_time, x_label = 'proton density')
create_histogram(earth_KP_index, earth_KPindex_data.iloc[:,0], CME1_shock1, CME2_end_time, x_label = 'KP index')

In [ ]:
create_histogram(earth_proton_temp, earth_DSCOVR_data.iloc[:,0], datetime.datetime(2018,8,16,tzinfo=datetime.timezone.utc), CME1_shock1, x_label = 'Proton Temperature')
create_histogram(earth_proton_flux, earth_GOES_data.iloc[:,0], datetime.datetime(2018,8,16,tzinfo=datetime.timezone.utc), CME1_shock1, x_label = 'proton flux')
create_histogram(earth_proton_density, earth_ACE_data.iloc[:,0], datetime.datetime(2018,8,16,tzinfo=datetime.timezone.utc), CME1_shock1, x_label = 'proton density')
create_histogram(earth_KP_index, earth_KPindex_data.iloc[:,0], datetime.datetime(2018,8,16,tzinfo=datetime.timezone.utc), CME1_shock1, x_label = 'KP index')

In [ ]:
create_histogram(earth_proton_temp, earth_DSCOVR_data.iloc[:,0], CME2_end_time, datetime.datetime(2018,9,10,tzinfo=datetime.timezone.utc), x_label = 'Proton Temperature')
create_histogram(earth_proton_flux, earth_GOES_data.iloc[:,0], CME2_end_time, datetime.datetime(2018,9,10,tzinfo=datetime.timezone.utc), x_label = 'proton flux')
create_histogram(earth_proton_density, earth_ACE_data.iloc[:,0], CME2_end_time, datetime.datetime(2018,9,10,tzinfo=datetime.timezone.utc), x_label = 'proton density')
create_histogram(earth_KP_index, earth_KPindex_data.iloc[:,0], CME2_end_time, datetime.datetime(2018,9,10,tzinfo=datetime.timezone.utc), x_label = 'KP index')

# New CME data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore

def create_histograms(file_path, start_end_times, data_columns):
    # Read CSV file
    data = pd.read_csv(file_path)

    # Ensure the first column is treated as datetime
    data.iloc[:, 0] = pd.to_datetime(data.iloc[:, 0])

    # Iterate through each pair of start and end times
    for start_time_str, end_time_str in start_end_times:
        # Convert strings to datetime objects
        start_time = pd.to_datetime(start_time_str)
        end_time = pd.to_datetime(end_time_str)

        # Filter data based on start and end times
        filtered_data = data[(data.iloc[:, 0] >= start_time) & (data.iloc[:, 0] <= end_time)]

        # Convert eV to K if applicable
        for column in data_columns:
            if 'eV' in column:
                filtered_data[column] = filtered_data[column] * 11604.525
                # Rename column to indicate the conversion
                filtered_data.rename(columns={column: column.replace('eV', 'K')}, inplace=True)

        # Remove extreme outliers using z-score
        numeric_columns = filtered_data[data_columns].select_dtypes(include='number')
        z_scores = zscore(numeric_columns)
        abs_z_scores = abs(z_scores)
        filtered_data = filtered_data[(abs_z_scores < 3).all(axis=1)]

        # Create histogram for each specified numeric column
        for column in data_columns:
            plt.hist(filtered_data[column], bins=30, alpha=0.7, label=column)
            plt.title(f'Histogram of {column} from {start_time} to {end_time}')
            plt.xlabel(column)
            plt.ylabel('Frequency')
            plt.legend()
            plt.show()


In [ ]:
create_histograms('/content/drive/MyDrive/mySEES/Proton_temp_SEES_2024/CME_2018/DSCOVR_temperature_Earth.csv', )